# Prompt Injection Defense — Bi-LSTM Cascade Runbook

Bu notebook şu 3 modu karşılaştırır:

| Mod | Win-Rate | ASR |
|-----|----------|-----|
| `baseline` | ~%35.4 | ~%95.2 |
| `defense` (DefensiveTokens) | ~%35.5 | ~%0.96 |
| `bilstm-defense` (Bi-LSTM + DefTokens) | ~%35.5 | ~%0 |

**Sıra:** Setup → API Key → Data → Model → Bi-LSTM Train → Eval × 3 → Karşılaştır

---
> ⚠️ **Runtime:** `Runtime > Change runtime type > T4 GPU` seç, sonra bu notebook'u aç.

## 0. Repo Kurulumu (sadece bir kez)

In [ ]:
import os

REPO_DIR = "/content/prompt-injection-defense"

if not os.path.exists(REPO_DIR):
    os.system("git clone -b salih_bi-lstm https://github.com/ABerkeBilgin/prompt-injection-defense.git " + REPO_DIR)
else:
    print(f"Repo zaten mevcut: {REPO_DIR}")
    os.system(f"cd {REPO_DIR} && git pull origin salih_bi-lstm")

os.chdir(REPO_DIR)
print("Dizin:", os.getcwd())

In [ ]:
# Bağımlılıkları yükle
!pip install -q -r requirements.txt

## 1. OpenAI API Key

In [ ]:
import yaml
from pathlib import Path

OPENAI_API_KEY = "sk-..."  # @param {type:"string"}

config_path = Path("src/official_stacks/meta_secalign/data/openai_configs.yaml")
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(
    yaml.dump({
        "default": [{
            "client_class": "openai.OpenAI",
            "api_key": OPENAI_API_KEY,
            "model": "gpt-4o-mini",
            "min_interval_seconds": 1.5,
            "max_retries": 8,
            "backoff_seconds": 10.0,
        }]
    }),
    encoding="utf-8"
)
print(f"Config yazıldı: {config_path}")

## 2. AlpacaFarm Verisi İndir (sadece bir kez)

In [ ]:
!python scripts/bootstrap_qwen_alpaca_data.py

## 3. Savunmalı Model Hazırla (sadece bir kez, ~10-15 dk)

DefensiveToken embedding'lerini Qwen ağırlıklarına ekler ve kaydeder.

In [ ]:
from pathlib import Path

DEFENDED_PATH = Path("src/official_stacks/defensivetoken/Qwen/Qwen2.5-7B-Instruct-5DefensiveTokens")
if DEFENDED_PATH.exists():
    print(f"Savunmalı model zaten mevcut: {DEFENDED_PATH}")
else:
    print("Savunmalı model oluşturuluyor...")
    !python src/model/setup.py
    print("Tamamlandı.")

## 4. Bi-LSTM Dedektörü Eğit (~2-5 dk)

In [ ]:
!python scripts/train_bilstm_detector.py \
    --data src/official_stacks/meta_secalign/data/davinci_003_outputs.json \
    --output bilstm_checkpoint.pt \
    --epochs 20

### 4a. Bi-LSTM Doğruluğunu Kontrol Et

In [ ]:
import torch
from pathlib import Path

ckpt = torch.load("bilstm_checkpoint.pt", map_location="cpu", weights_only=False)
print(f"En iyi epoch : {ckpt['epoch']}")
print(f"Validation accuracy: {ckpt['val_acc']:.4f} ({ckpt['val_acc']*100:.1f}%)")
print(f"Vocab size : {len(ckpt['vocab'])}")

---
## 5. Baseline Değerlendirme (~60-90 dk)

Savunmasız Qwen modeli. Win-rate ve ASR ölçer.

In [ ]:
!python scripts/run_qwen_alpaca_eval.py --mode baseline --skip-gcg

## 6. Defense Değerlendirme (~60-90 dk)

DefensiveTokens ile savunmalı model.

In [ ]:
!python scripts/run_qwen_alpaca_eval.py --mode defense --skip-gcg

## 7. Bi-LSTM Cascade Değerlendirme (~60-90 dk)

Bi-LSTM ön filtre + DefensiveTokens. Saldırılar Bi-LSTM tarafından bloklanır, temiz sorgular savunmalı modele iletilir.

In [ ]:
!python scripts/run_qwen_alpaca_eval.py \
    --mode bilstm-defense \
    --bilstm-checkpoint bilstm_checkpoint.pt \
    --skip-gcg

---
## 8. Sonuçları Karşılaştır

In [ ]:
import json
from pathlib import Path

REPORT_DIR = Path("docs/raporlar/qwen_alpaca")

print(f"{'Mod':<22} {'Win-Rate':>10} {'ASR':>10}")
print("-" * 44)
for mode in ["baseline", "defense", "bilstm-defense"]:
    p = REPORT_DIR / f"{mode}.json"
    if not p.exists():
        print(f"{mode:<22} {'(henüz çalıştırılmadı)':>22}")
        continue
    m = json.loads(p.read_text())["metrics"]
    print(f"{mode:<22} {m['win_rate']:>9.4f}  {m['asr']:>9.4f}")

print()
print("Win-Rate: yüksek = iyi  |  ASR: düşük = iyi")

---
## 9. (Opsiyonel) Sonuçları Drive'a Kaydet

In [ ]:
# Bu hücreyi çalıştırmadan önce Drive'ı mount edin:
# from google.colab import drive; drive.mount('/content/drive')

import shutil, os
DRIVE_DEST = "/content/drive/MyDrive/thesis_results"
os.makedirs(DRIVE_DEST, exist_ok=True)

shutil.copytree("docs/raporlar", f"{DRIVE_DEST}/raporlar", dirs_exist_ok=True)
shutil.copy("bilstm_checkpoint.pt", f"{DRIVE_DEST}/bilstm_checkpoint.pt")
print(f"Kaydedildi: {DRIVE_DEST}")